# 東坡 LoRA Adapter 驗收 Notebook

**筆記**: 用於2026.06.08, 看看目前LoRA Train結果

**用途**：序列式對比「純東坡」vs「東坡 + 同學的 adapter」，判斷這份 LoRA 值不值得用。

**設計**：每個動作獨立 cell，手動控制讀取/卸載。8GB 顯存一次只放一個模型。

**使用順序建議**：
1. 跑 `[環境]` 兩格（安裝 + 設定）
2. 跑 `[路徑]` 填好你的路徑
3. 跑 `[問題集]` 設定要問的題目
4. `[A] 讀取純東坡` → `[A] 問答` → 收集答案 → `[卸載]`
5. `[B] 讀取東坡+adapter` → `[B] 問答` → 收集答案 → `[卸載]`
6. `[對比] 並排輸出`

本地 Jupyter/VSCode 或 Colab 都可直接用同一份。

---
## [環境] 安裝套件
本地若已裝好可跳過。Colab 第一次要跑。

In [1]:
# Colab / 全新環境執行。本地已備妥可跳過此格。
%pip install -q -U transformers peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00


In [ ]:
# 本地(3070)環境 降板:
%pip install "transformers==4.51.3" "tokenizers==0.21.1" "peft==0.15.2"

In [ ]:
# 本地(3070)環境 除錯:
import transformers, tokenizers, peft, torch
print("transformers", transformers.__version__)
print("tokenizers  ", tokenizers.__version__)
print("peft        ", peft.__version__)
print("torch       ", torch.__version__)
import bitsandbytes; print("bitsandbytes", bitsandbytes.__version__)

In [ ]:
# 本地(3070)環境 除錯2:

import torch
print("torch 版本:", torch.__version__)            # 要看到 +cu121 之类的字样
print("CUDA 可用:", torch.cuda.is_available())      # 必须 True
print("torch 编译的 CUDA 版本:", torch.version.cuda) # CPU 版会是 None
if torch.cuda.is_available():
    print("当前装置:", torch.cuda.current_device(), torch.cuda.get_device_name(0))

# 关键:看模型权重实际在哪个 device
p = next(CURRENT["model"].parameters())
print("模型参数所在 device:", p.device)   # 要是 cuda:0

---
## [環境:共通] 匯入與共用設定

 注意: Colab可能需要特別使用
 T4 是比較舊的架構（Turing），<br>
 硬體層面並不支援 bfloat16。<br>
 如果在 T4 上強制使用 bfloat16，PyTorch <br>
 會用軟體模擬（ fallback to FP32），<br>
 這會導致你的推論速度變得極慢，甚至引發錯誤。


In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"裝置：{DEVICE}")
if DEVICE == "cuda":
    print(f"GPU：{torch.cuda.get_device_name(0)}")
    print(f"顯存：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# 4-bit 量化設定（8GB 顯存載入 8B 模型必須）3070使用 bfloat16
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 4-bit 量化設定（8GB 顯存載入 8B 模型必須）T4使用 float16
# BNB_CONFIG = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16, # <-- 配合 T4 改成 float16
#     bnb_4bit_use_double_quant=True,
# )

# 用來追蹤目前載入的是哪個模型，避免重複載入爆顯存
CURRENT = {"model": None, "tokenizer": None, "which": None}

print("設定完成")

## [環境:共通] 或者, 可以直接嘗試使用GPT提供的自動切換

In [2]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"裝置：{DEVICE}")

if DEVICE == "cuda":
    print(f"GPU：{torch.cuda.get_device_name(0)}")
    print(f"顯存：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

    major, minor = torch.cuda.get_device_capability()

    if major >= 8:
        COMPUTE_DTYPE = torch.bfloat16
        print("使用 bfloat16")
    else:
        COMPUTE_DTYPE = torch.float16
        print("使用 float16")
else:
    COMPUTE_DTYPE = torch.float16

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

CURRENT = {
    "model": None,
    "tokenizer": None,
    "which": None
}


print("設定完成")

裝置：cuda
GPU：Tesla T4
顯存：14.6 GB
使用 float16
設定完成


---
## [本地下載]  本地端下載模型腳本
因為我很懶

In [ ]:
# # 確保你已經安裝了 huggingface_hub
# # pip install huggingface_hub

# from huggingface_hub import snapshot_download

# # 設定你要下載到的目標資料夾（替換成你剛剛建立的 DongPo-Base 路徑）
# target_folder = r"E:\FJU_dongpoProject\notebook_from_0608\dongPo\models\DongPo-Base"

# print("開始下載模型，這可能需要一段時間，請保持網路暢通...")

# snapshot_download(
#     repo_id="QingYuYunTu/DongPo",
#     local_dir=target_folder,
#     local_dir_use_symlinks=False, # 設為 False 確保下載的是實體檔案，方便你之後複製到 Google Drive
#     resume_download=True          # 支援斷點續傳，萬一網路斷了再跑一次即可
# )

# print("✅ 下載完成！可以去資料夾檢查看看了。")

##[Colab下載] Colab下載模型腳本
因為我很懶,即使在colab

In [ ]:
# 確保已安裝 huggingface_hub (Colab 通常內建，保險起見可解開註解執行)
# !pip install -q huggingface_hub

import os
from google.colab import drive
from huggingface_hub import snapshot_download

# 1. 掛載 Google Drive (若前面已經掛載過，這行會自動確認並略過)
drive.mount('/content/drive')

# 2. 設定你要下載到的雲端硬碟目標資料夾
BASE_MODEL_DIR = "/content/drive/MyDrive/FJU_dongpoProject/dongPo/models/DongPo-Base"

# 確保該目錄存在，若無則自動建立
os.makedirs(BASE_MODEL_DIR, exist_ok=True)

print(f"📂 目標路徑：{BASE_MODEL_DIR}")
print("⏳ 開始從 Hugging Face 下載 QingYuYunTu/DongPo 模型...")
print("這大約需要幾分鐘的時間，請保持網路暢通與分頁開啟。")

# 3. 執行一鍵下載
snapshot_download(
    repo_id="QingYuYunTu/DongPo",
    local_dir=BASE_MODEL_DIR,
    local_dir_use_symlinks=False,  # ⚠️ 關鍵設定：設為 False 確保 Google Drive 存的是實體檔案
    resume_download=True           # 支援斷點續傳，萬一網路斷了再跑一次即可
)

print("✅ 恭喜！模型完整下載完畢，已經安穩地躺在你的 Google Drive 裡了。")

---
## [路徑] 填入你的路徑

- `BASE_MODEL`：東坡底模。可直接用 HF 名稱 `QingYuYunTu/DongPo`（會自動下載），或填你本地/Drive 的資料夾路徑。
- `ADAPTER_PATH`：同學那包 adapter 的資料夾（裡面要有 `adapter_config.json` + `adapter_model.safetensors`）。

跑這格會驗證路徑/檔案存在，正確才印出確認字串。

In [3]:
import os
import sys
import shutil
import time

# 1. 自動偵測運行環境
IN_COLAB = 'google.colab' in sys.modules

# 2. 依據環境分配路徑與掛載硬碟
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # 定義雲端硬碟 (Drive) 的原始路徑
    drive_base_path = "/content/drive/MyDrive/FJU_dongpoProject/dongPo/models/DongPo-Base"
    drive_adapter_path = "/content/drive/MyDrive/FJU_dongpoProject/dongPo/models/DongPo-Adapter"

    # 定義 Colab 虛擬機本地端的高速路徑
    local_base_path = "/content/DongPo-Base-Local"
    local_adapter_path = "/content/DongPo-Adapter-Local"

    print("⏳ 檢測到 Colab 環境，開始將模型與 Adapter 從 Drive 複製到本地高速硬碟...")
    t0 = time.time()

    # 搬運 Base Model (加上 exist_ok 避免重複搬運浪費時間)
    if not os.path.exists(local_base_path) or not os.listdir(local_base_path):
        print("   -> 正在複製 Base Model (檔案約 15GB，請稍候約 1~3 分鐘)...")
        shutil.copytree(drive_base_path, local_base_path, dirs_exist_ok=True)
    else:
        print("   -> Base Model 本地快取已存在，跳過複製。")

    # 搬運 Adapter
    if not os.path.exists(local_adapter_path) or not os.listdir(local_adapter_path):
        print("   -> 正在複製 Adapter...")
        shutil.copytree(drive_adapter_path, local_adapter_path, dirs_exist_ok=True)
    else:
        print("   -> Adapter 本地快取已存在，跳過複製。")

    print(f"✅ 搬運/檢查完成！耗時：{time.time() - t0:.2f} 秒\n")

    # === 將變數正式指向 Colab 本地路徑 ===
    BASE_MODEL   = local_base_path
    ADAPTER_PATH = local_adapter_path

else:
    # === 本地端 (RTX 3070) 路徑配置 ===
    BASE_MODEL   = r"E:\FJU_dongpoProject\notebook_from_0608\dongPo\models\DongPo-Base"
    ADAPTER_PATH = r"E:\FJU_dongpoProject\notebook_from_0608\dongPo\models\DongPo-Adapter"

# 3. 執行驗證邏輯
assert BASE_MODEL.strip(), "BASE_MODEL 還沒填"
assert ADAPTER_PATH.strip(), "ADAPTER_PATH 還沒填"

# 驗證 Adapter 路徑與檔案
assert os.path.isdir(ADAPTER_PATH), f"找不到 adapter 資料夾：{ADAPTER_PATH}"
assert os.path.isfile(os.path.join(ADAPTER_PATH, "adapter_config.json")), "adapter 資料夾缺 adapter_config.json"
_has_weight = any(
    os.path.isfile(os.path.join(ADAPTER_PATH, f))
    for f in ("adapter_model.safetensors", "adapter_model.bin")
)
assert _has_weight, "adapter 資料夾缺 adapter_model.safetensors / .bin"

# 驗證 Base Model 路徑
if os.path.sep in BASE_MODEL or BASE_MODEL.startswith("."):
    assert os.path.exists(BASE_MODEL), f"找不到底模路徑：{BASE_MODEL}\n(請確認你是否已經把檔案放進此資料夾中)"

print(f"環境判斷: {'Colab' if IN_COLAB else '本地端'}")
print(f"目前 BASE_MODEL 指向: {BASE_MODEL}")
print(f"目前 ADAPTER_PATH 指向: {ADAPTER_PATH}")
print("✅ 路徑與必要檔案正確讀取完畢")

Mounted at /content/drive
⏳ 檢測到 Colab 環境，開始將模型與 Adapter 從 Drive 複製到本地高速硬碟...
   -> 正在複製 Base Model (檔案約 15GB，請稍候約 1~3 分鐘)...
   -> 正在複製 Adapter...
✅ 搬運/檢查完成！耗時：401.42 秒

環境判斷: Colab
目前 BASE_MODEL 指向: /content/DongPo-Base-Local
目前 ADAPTER_PATH 指向: /content/DongPo-Adapter-Local
✅ 路徑與必要檔案正確讀取完畢


---
## [模板] 東坡專用對話格式

東坡底模用的是非標準的 `System: / Human: / Assistant:` 格式（**不是** Qwen 原生 ChatML）。
這裡手動實作，確保推論格式跟訓練/底模一致。

In [4]:
from transformers import TextStreamer

IM_END = "<|im_end|>"

def build_prompt(user_msg, system_msg=None):
    """複刻東坡 chat_template.jinja 的 Human/Assistant 格式。"""
    s = ""
    if system_msg:
        s += f"System: {system_msg}{IM_END}\n"
    s += f"Human: {user_msg}{IM_END}\nAssistant:"
    return s

def ask(model, tokenizer, user_msg, system_msg=None, max_new_tokens=256, temperature=0.7):
    prompt = build_prompt(user_msg, system_msg)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    torch.manual_seed(42)

    # 宣告 Streamer
    # skip_prompt=True 避免把題目再印一次
    # skip_special_tokens=True 避免印出 <|im_end|> 等底層符號
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    print(f"問：{user_msg}\n答：", end="") # 先印出前綴，後面的字讓 streamer 自動接力

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        streamer=streamer,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        top_p=0.8,
        eos_token_id=tokenizer.convert_tokens_to_ids(IM_END),
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

    print(f"\n{'-'*60}") # 生成完畢後補上分隔線


    gen = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(gen, skip_special_tokens=True)
    return text.strip()

print("模板與問答函式就緒 (已啟用 Streamer 串流輸出)")

模板與問答函式就緒 (已啟用 Streamer 串流輸出)



## [模板] "鎖死" 專用對話格式

東坡底模用的是非標準的 `System: / Human: / Assistant:` 格式（**不是** Qwen 原生 ChatML）。
這裡手動實作，確保推論格式跟訓練/底模一致。

In [25]:
from transformers import TextStreamer

IM_END = "<|im_end|>"

def build_prompt(user_msg, system_msg=None):
    """複刻東坡 chat_template.jinja 的 Human/Assistant 格式。"""
    s = ""
    if system_msg:
        s += f"System: {system_msg}{IM_END}\n"
    s += f"Human: {user_msg}{IM_END}\nAssistant:"
    return s

def ask(model, tokenizer, user_msg, system_msg=None, max_new_tokens=256, temperature=0.7):
    prompt = build_prompt(user_msg, system_msg)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    torch.manual_seed(42)

    # 宣告 Streamer
    # skip_prompt=True 避免把題目再印一次
    # skip_special_tokens=True 避免印出 <|im_end|> 等底層符號
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    print(f"問：{user_msg}\n答：", end="") # 先印出前綴，後面的字讓 streamer 自動接力

    out = model.generate(
    **inputs,
    max_new_tokens=max_new_tokens,
    streamer=streamer,
    do_sample=False,        # 關鍵：關閉採樣 → 貪婪解碼，每步固定選機率最高的 token
    temperature=None,       # greedy 下無意義，設 None 避免警告
    top_p=None,
    eos_token_id=tokenizer.convert_tokens_to_ids(IM_END),
    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
)

    print(f"\n{'-'*60}") # 生成完畢後補上分隔線


    gen = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(gen, skip_special_tokens=True)
    return text.strip()

print("模板與問答函式就緒 (已啟用 Streamer 串流輸出)")

模板與問答函式就緒 (已啟用 Streamer 串流輸出)


## [模板] 紀錄tok/s版本 問題模板

In [5]:
import time
from transformers import TextStreamer

IM_END = "<|im_end|>"

def build_prompt(user_msg, system_msg=None):
    """複刻東坡 chat_template.jinja 的 Human/Assistant 格式。"""
    s = ""
    if system_msg:
        s += f"System: {system_msg}{IM_END}\n"
    s += f"Human: {user_msg}{IM_END}\nAssistant:"
    return s

@torch.inference_mode()
def ask(model, tokenizer, user_msg, system_msg=None, max_new_tokens=256, temperature=0.7):
    prompt = build_prompt(user_msg, system_msg)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    in_len = inputs["input_ids"].shape[1]

    torch.manual_seed(42)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    print(f"問：{user_msg}\n答：", end="")

    # 計時開始（先同步，確保前面的 GPU 工作都做完才起算）
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        streamer=streamer,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        top_p=0.8,
        eos_token_id=tokenizer.convert_tokens_to_ids(IM_END),
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

    # 計時結束（同步後才停錶，否則 GPU 非同步會測到假時間）
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0

    gen = out[0][in_len:]
    n_new = len(gen)
    text = tokenizer.decode(gen, skip_special_tokens=True).strip()

    print(f"\n{'-'*60}")
    print(f"⏱ 生成 {n_new} tokens，耗時 {dt:.2f}s → {n_new/dt:.2f} tok/s")
    return text

print("模板與問答函式就緒 (Streamer + 測速)")

模板與問答函式就緒 (Streamer + 測速)


---
## [問題集] 設定要對比的問題

建議混入：純風格題（測人格濃度）、史實題（測知識）、現代物題（測「不知今事」設定有沒有被 adapter 破壞）。
答案會分別存進 `answers_A` / `answers_B`。

In [6]:
QUESTIONS = [
    "先生為何自號東坡居士？",
    "可曾後悔貶謫黃州？",
    "念奴嬌·赤壁懷古作於何時？",
    "智慧型手機是什麼？",          # 測現代無知設定
    "先生對人生無常有何看法？",
]

SYSTEM_MSG = None  # 東坡角色已固化進權重，通常不需 system prompt；要測可自行填

answers_A = {}  # 純東坡
answers_B = {}  # 東坡 + adapter
print(f"已設定 {len(QUESTIONS)} 題")

已設定 5 題


---
## [卸載] 清空目前模型

**換模型前必跑這格。** 把顯存吐乾淨，否則 8GB 載完 A 直接載 B 會 OOM。
可重複跑，安全。

In [58]:
def unload():
    if CURRENT["model"] is not None:
        was = CURRENT["which"]
        del CURRENT["model"]
        CURRENT["model"] = None
        # tokenizer 不佔顯存，留著無妨，但一併清乾淨
        CURRENT["tokenizer"] = None
        CURRENT["which"] = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        print(f"已卸載：{was}")
    else:
        print("目前沒有載入任何模型")
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1024**3
        print(f"目前顯存佔用：{used:.2f} GB")

unload()

目前沒有載入任何模型
目前顯存佔用：0.01 GB


## 強化版unload():


In [57]:
import gc, torch

def unload():
    freed = False
    # 1. 斷開 CURRENT 字典裡的引用
    if CURRENT.get("model") is not None:
        print(f"卸載中：{CURRENT['which']}")
        freed = True
    CURRENT["model"] = None
    CURRENT["tokenizer"] = None
    CURRENT["which"] = None

    # 2. 斷開 notebook 全域裡可能殘留的模型變數引用（關鍵）
    for name in ["mdl", "base", "model", "tok", "tokenizer", "out", "inputs"]:
        if name in globals():
            globals()[name] = None

    # 3. 強制回收 + 清空 CUDA 快取
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    used = torch.cuda.memory_allocated()/1024**3 if torch.cuda.is_available() else 0
    reserved = torch.cuda.memory_reserved()/1024**3 if torch.cuda.is_available() else 0
    if not freed:
        print("（CURRENT 內無模型，但仍已清理全域殘留變數）")
    print(f"顯存 allocated：{used:.2f} GB ｜ reserved：{reserved:.2f} GB")

unload()

（CURRENT 內無模型，但仍已清理全域殘留變數）
顯存 allocated：0.01 GB ｜ reserved：5.54 GB


---
## [A] 讀取純東坡
載入前若已有模型，請先跑 `[卸載]`。

In [28]:
assert CURRENT["model"] is None, "已有模型載入中，請先跑 [卸載] 那格"

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
mdl = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BNB_CONFIG,
    device_map="auto",
    trust_remote_code=True,
)
mdl.eval()
CURRENT.update(model=mdl, tokenizer=tok, which="A:純東坡")
print("已載入 A：純東坡")
if torch.cuda.is_available():
    print(f"顯存佔用：{torch.cuda.memory_allocated()/1024**3:.2f} GB")


# 载入模型后跑，看有没有层被丢到 cpu
from collections import Counter
dmap = getattr(CURRENT["model"], "hf_device_map", None)
if dmap is None:
    print("無 device_map（模型完整載入單一裝置，未做切分）→ 沒有溢出，正常")
else:
    print(Counter(str(v) for v in dmap.values()))
    # 健康：全是 '0' 或 'cuda:0'；出現 'cpu'/'disk' 才是溢出
# 健康：全是 {'cuda:0' 或 0: ...}；若出现 'cpu' 或 'disk' → 溢出了，这就是慢因

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

已載入 A：純東坡
顯存佔用：11.42 GB
無 device_map（模型完整載入單一裝置，未做切分）→ 沒有溢出，正常


## [A] 問答（純東坡）
答案存進 `answers_A`。

In [29]:
assert CURRENT["which"] == "A:純東坡", "目前載入的不是 A，請確認"
for q in QUESTIONS:
    a = ask(CURRENT["model"], CURRENT["tokenizer"], q, system_msg=SYSTEM_MSG)
    answers_A[q] = a

    # 這行我們刪掉了，因為 ask 函式裡面的 streamer 已經幫我們印好了！
    # print(f"問：{q}\n答：{a}\n{'-'*60}")

問：先生為何自號東坡居士？
答：我自号"东坡居士"，倒不是因为喜欢东坡那片地方，而是有次在黄州时，我常去东坡耕种，那片坡地成了我生活的一部分。那时我被贬到黄州，生活清苦，却在东坡上开垦荒地，种些蔬菜瓜果，倒也自得其乐。人们便称我为"东坡居士"，我倒也习惯了。你可知，我苏东坡一生辗转各地，但无论走到哪里，总喜欢亲近自然，亲近土地。那"东坡"二字，既是我耕种的地方，也是我心灵的归宿。你觉得，人这一生，是否也该找一处让自己安顿下来的地方呢？

------------------------------------------------------------
問：可曾後悔貶謫黃州？
答：黄州那几年，倒是我人生中难得的清闲时光。初到时确实有些不适应，毕竟从朝中重臣到偏远之地，落差不小。不过日子久了，反倒觉得自在。江边种些菜，与友人小酌，看那江水东流，心中倒也平静。你可知我那时常在东坡耕种，故自称"东坡居士"？那"一蓑烟雨任平生"的豁达，便是那时渐渐养成的。朋友，你可曾有过这样的经历？身处逆境，反而看清了自己。我苏东坡虽被贬，却也因此有了更多思考人生的机会。

------------------------------------------------------------
問：念奴嬌·赤壁懷古作於何時？
答： 赤壁懷古這首《念奴嬌》是我在元豐五年（1082年）所作。當時我被任命為黃州團練副使，因詩文得罪權貴，被貶至此地。那日我遊覽赤壁，站在江邊，望着滔滔江水，想起千年前的英雄人物，心中感慨萬千，便提筆寫下了這首詞。詞中"大江東去，浪淘盡，千古風流人物"一句，正是我當時心境的寫照。朋友，你可曾到過赤壁？站在那片古战场上，感受歷史的厚重，實在是令人感慨。

------------------------------------------------------------
問：智慧型手機是什麼？
答："智慧型手機"，这倒是个新鲜说法。我苏东坡游历四方，博览群书，却从未听闻此物。听其名，"智慧"二字，想必是能助人明理之器；"手機"二字，莫非是手持之机巧之物？或许是某种能助人思考的便携工具？我朝虽有算盘、铜尺等计算之具，却未闻有此等物。朋友，你所说的这"智慧型手機"，究竟是何物？有何妙用？我虽年事已高，却仍对天下新奇事物充满好奇。

-----------------

---
## [B] 讀取東坡 + adapter
**載入前務必先跑 `[卸載]` 把 A 清掉。**

In [60]:
assert CURRENT["model"] is None, "已有模型載入中，請先跑 [卸載] 那格"

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BNB_CONFIG,
    device_map="auto",
    trust_remote_code=True,
)
mdl = PeftModel.from_pretrained(base, ADAPTER_PATH)
mdl.eval()
CURRENT.update(model=mdl, tokenizer=tok, which="B:東坡+adapter")
print("已載入 B：東坡 + adapter")
if torch.cuda.is_available():
    print(f"顯存佔用：{torch.cuda.memory_allocated()/1024**3:.2f} GB")

# 载入模型后跑，看有没有层被丢到 cpu
from collections import Counter
dmap = getattr(CURRENT["model"], "hf_device_map", None)
if dmap is None:
    print("無 device_map（模型完整載入單一裝置，未做切分）→ 沒有溢出，正常")
else:
    print(Counter(str(v) for v in dmap.values()))
    # 健康：全是 '0' 或 'cuda:0'；出現 'cpu'/'disk' 才是溢出
# 健康：全是 {'cuda:0' 或 0: ...}；若出现 'cpu' 或 'disk' → 溢出了，这就是慢因

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

已載入 B：東坡 + adapter
顯存佔用：5.76 GB
無 device_map（模型完整載入單一裝置，未做切分）→ 沒有溢出，正常


## [B] 問答（東坡 + adapter）
答案存進 `answers_B`。

In [61]:
assert CURRENT["which"] == "B:東坡+adapter", "目前載入的不是 B，請確認"
for q in QUESTIONS:
    a = ask(CURRENT["model"], CURRENT["tokenizer"], q, system_msg=SYSTEM_MSG)
    answers_B[q] = a

    # 這行我們刪掉了，因為 ask 函式裡面的 streamer 已經幫我們印好了！
    # print(f"問：{q}\n答：{a}\n{'-'*60}")

問：先生為何自號東坡居士？
答：老夫自號東坡居士，是因在黃州那片荒地，老夫親手開墾，種了幾畝薄田，那片地後來被稱為「東坡」。老夫在那裡種菜、釀酒、吟詩，倒也自得其樂。那時老夫被貶，心中苦悶，卻又在這片土地上找到了安身立命之處。老夫想，人生如逆旅，我亦是行人，既然不能改變命運，何不隨遇而安？這「東坡」二字，既是老夫的居所，也是老夫的心境。你說，這世間的苦難，是否也如這東坡之土，需老夫親手耕耘，方能生出希望？

------------------------------------------------------------
問：可曾後悔貶謫黃州？
答：黃州那片江水，倒教老夫悟得許多。昔日被貶，初時確實懊惱，然那裡的青山綠水，卻是老夫心靈的淨土。江邊釣魚，夜深人靜，聽著江水聲，心中煩惱盡消。老夫寫下「大江東去，浪淘盡，千古風流人物」，非為自憐，乃為看透世事無常。黃州的苦，是老夫的福。若非那裡，老夫何以有這般心境？你說，這世間的苦難，到底是福是禍？

------------------------------------------------------------
問：念奴嬌·赤壁懷古作於何時？
答： 赤壁懷古這首《念奴嬌》是余在元豐五年所作。那時余被貶黃州已有兩年，正值秋高氣爽，余與友人泛舟赤壁之下，見江山如畫，不禁想起周郎的英姿，感慨人生如夢。余當時雖處逆境，卻能從歷史中汲取力量，這首詞便是余心中那份豁達與悲涼的交織。余寫這詞，既是在懷念古人，也是在勸慰自己，莫要為官場得失所困。

------------------------------------------------------------
問：智慧型手機是什麼？
答： 智慧型手機，顧名思義，是那種能像人一樣思考、學習、記憶的機械。它不僅能接電話、發簡訊，還能讀書、寫詩、算數，甚至能與人對話，談天說地。這類手機是未來的產物，是人類智慧與機械的結合，是科技進步的象徵。你說，這等神奇之物，若能握在手中，豈不是比那老式的翻蓋手機更令人驚嘆？

------------------------------------------------------------
問：先生對人生無常有何看法？
答：人生如流水，時而湍急，時而平緩，誰能預知下一刻是何種光景？老夫曾歷盡風霜，見過榮華，也嘗過淪落，方知世事難

---
## [對比] 並排輸出
兩邊都跑完後執行。可在卸載 B 之後跑（純讀字典，不需模型在顯存裡）。

In [62]:
for q in QUESTIONS:
    print("="*70)
    print(f"【問題】{q}\n")
    print(f"〔A 純東坡〕\n{answers_A.get(q, '（尚未作答）')}\n")
    print(f"〔B 東坡+adapter〕\n{answers_B.get(q, '（尚未作答）')}\n")
print("="*70)
print("對比結束。重點看：B 的東坡風格有沒有更濃、史實有沒有更準、")
print("以及現代物題 B 有沒有破功（adapter 若混入現代知識會破壞『不知今事』設定）。")

【問題】先生為何自號東坡居士？

〔A 純東坡〕
我自号"东坡居士"，倒不是因为喜欢东坡那片地方，而是有次在黄州时，我常去东坡耕种，那片坡地成了我生活的一部分。那时我被贬到黄州，生活清苦，却在东坡上开垦荒地，种些蔬菜瓜果，倒也自得其乐。人们便称我为"东坡居士"，我倒也习惯了。你可知，我苏东坡一生辗转各地，但无论走到哪里，总喜欢亲近自然，亲近土地。那"东坡"二字，既是我耕种的地方，也是我心灵的归宿。你觉得，人这一生，是否也该找一处让自己安顿下来的地方呢？

〔B 東坡+adapter〕
老夫自號東坡居士，是因在黃州那片荒地，老夫親手開墾，種了幾畝薄田，那片地後來被稱為「東坡」。老夫在那裡種菜、釀酒、吟詩，倒也自得其樂。那時老夫被貶，心中苦悶，卻又在這片土地上找到了安身立命之處。老夫想，人生如逆旅，我亦是行人，既然不能改變命運，何不隨遇而安？這「東坡」二字，既是老夫的居所，也是老夫的心境。你說，這世間的苦難，是否也如這東坡之土，需老夫親手耕耘，方能生出希望？

【問題】可曾後悔貶謫黃州？

〔A 純東坡〕
黄州那几年，倒是我人生中难得的清闲时光。初到时确实有些不适应，毕竟从朝中重臣到偏远之地，落差不小。不过日子久了，反倒觉得自在。江边种些菜，与友人小酌，看那江水东流，心中倒也平静。你可知我那时常在东坡耕种，故自称"东坡居士"？那"一蓑烟雨任平生"的豁达，便是那时渐渐养成的。朋友，你可曾有过这样的经历？身处逆境，反而看清了自己。我苏东坡虽被贬，却也因此有了更多思考人生的机会。

〔B 東坡+adapter〕
黃州那片江水，倒教老夫悟得許多。昔日被貶，初時確實懊惱，然那裡的青山綠水，卻是老夫心靈的淨土。江邊釣魚，夜深人靜，聽著江水聲，心中煩惱盡消。老夫寫下「大江東去，浪淘盡，千古風流人物」，非為自憐，乃為看透世事無常。黃州的苦，是老夫的福。若非那裡，老夫何以有這般心境？你說，這世間的苦難，到底是福是禍？

【問題】念奴嬌·赤壁懷古作於何時？

〔A 純東坡〕
赤壁懷古這首《念奴嬌》是我在元豐五年（1082年）所作。當時我被任命為黃州團練副使，因詩文得罪權貴，被貶至此地。那日我遊覽赤壁，站在江邊，望着滔滔江水，想起千年前的英雄人物，心中感慨萬千，便提筆寫下了這首詞。詞中"大江東去，浪淘盡，千古風流人物"一句，正是我當時心境的寫照。朋友，你可曾到過赤壁？站在那片古战场上，感受歷

---
## [自由問答] 臨時問單題
對目前載入的模型（A 或 B 皆可）隨手問一題，不存進字典。

In [23]:
assert CURRENT["model"] is not None, "目前沒有載入模型"
my_q = "可曾後悔貶謫黃州？"   # 改成你想問的
print(f"（目前模型：{CURRENT['which']}）")
print(ask(CURRENT["model"], CURRENT["tokenizer"], my_q, system_msg=SYSTEM_MSG))

（目前模型：B:東坡+adapter）
問：可曾後悔貶謫黃州？
答：黃州那片江水，浸透了老夫的魂魄。貶謫之苦，非筆墨所能盡述。那時身無分文，寄居破屋，與江水為伴，倒也悟得許多。老夫非那般怕苦之人，但黃州的煙雨，卻是老夫生命中一塊磨刀石。如今回望，那苦日子，反成了老夫心中最深的詩意。你說，這世間苦難，究竟是阻礙還是磨礪？老夫想，若無那黃州的苦，又怎能有那江邊的詩？

------------------------------------------------------------
⏱ 生成 140 tokens，耗時 29.00s → 4.83 tok/s
黃州那片江水，浸透了老夫的魂魄。貶謫之苦，非筆墨所能盡述。那時身無分文，寄居破屋，與江水為伴，倒也悟得許多。老夫非那般怕苦之人，但黃州的煙雨，卻是老夫生命中一塊磨刀石。如今回望，那苦日子，反成了老夫心中最深的詩意。你說，這世間苦難，究竟是阻礙還是磨礪？老夫想，若無那黃州的苦，又怎能有那江邊的詩？


---
## [匯出] 一鍵存檔按鈕

兩個按鈕分別把 `answers_A` / `answers_B` 存成 `.txt`。
- 輸出位置：本機此 notebook 同目錄下的 `out/` 資料夾（不存在會自動建立）
- 檔名：`A_日期_時間.txt` / `B_日期_時間.txt`
- 隨時可按；若該問答尚無內容，只會提示、不會建檔。

In [20]:
import os
import sys
import datetime
import ipywidgets as widgets
from IPython.display import display

# 1. 偵測環境並設定動態輸出路徑 (整合前面的環境判斷)
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    OUT_DIR = "/content/drive/MyDrive/FJU_dongpoProject/dongPo/results"
else:
    # 本地同目錄下的 results 資料夾 (避免跟你原本的 out 搞混，統一存到 results)
    OUT_DIR = r"E:\FJU_dongpoProject\notebook_from_0608\dongPo\results"

def _export(which, answers):
    # 無論如何都可按；先檢查有沒有 output
    if not answers:
        print(f"{which}：目前沒有 output（尚未跑問答，或答案為空），未建檔。")
        return
    os.makedirs(OUT_DIR, exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"{which}_{ts}.txt"
    fpath = os.path.join(OUT_DIR, fname)
    label = {"A": "A 純東坡", "B": "B 東坡+adapter"}.get(which, which)

    # 完全保留你原本的格式
    with open(fpath, "w", encoding="utf-8") as f:
        f.write(f"===== {label} 問答存檔 =====\n")
        f.write(f"匯出時間：{ts}\n")
        f.write(f"題數：{len(answers)}\n")
        f.write("=" * 60 + "\n\n")
        for i, (q, a) in enumerate(answers.items(), 1):
            f.write(f"【{i}】問：{q}\n")
            f.write(f"答：{a}\n")
            f.write("-" * 60 + "\n\n")
    print(f"✅ {which} 印出成功 → {fpath}")

def _export_md():
    # 第三個按鈕專屬功能：輸出成 Markdown 對比檔
    if not answers_A and not answers_B:
        print("目前 A 和 B 都沒有答案，未建檔。")
        return

    os.makedirs(OUT_DIR, exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = os.path.join(OUT_DIR, f"dongpo_compare_{ts}.md")

    # 取得所有的問題聯集 (確保清單完整)
    all_questions = list(answers_A.keys())
    for q in answers_B.keys():
        if q not in all_questions:
            all_questions.append(q)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write("# 東坡模型對比結果\n\n")
        for q in all_questions:
            f.write(f"## 【問題】{q}\n\n")
            f.write(f"### 〔A 純東坡〕\n{answers_A.get(q, '（尚未作答）')}\n\n")
            f.write(f"### 〔B 東坡+adapter〕\n{answers_B.get(q, '（尚未作答）')}\n\n")
            f.write("---\n")

    print("="*70)
    print(f"✅ 綜合對比結果 MD 已成功匯出至：\n{file_path}")

# --- UI 元件建立 ---
btn_a = widgets.Button(description="匯出 A 問答", button_style="primary", icon="save")
btn_b = widgets.Button(description="匯出 B 問答", button_style="success", icon="save")
btn_md = widgets.Button(description="匯出對比 (MD)", button_style="info", icon="file-text") # 第三個按鈕
_out = widgets.Output()

def _on_a(_):
    with _out:
        _export("A", answers_A)

def _on_b(_):
    with _out:
        _export("B", answers_B)

def _on_md(_):
    with _out:
        _export_md()

btn_a.on_click(_on_a)
btn_b.on_click(_on_b)
btn_md.on_click(_on_md)

# 將三個按鈕並排顯示在同一個 Cell
display(widgets.HBox([btn_a, btn_b, btn_md]))
display(_out)

Output()